### 5.2.1 用 `scipy.optimize` 模块求解  
SciPy 的 `scipy.optimize` 模块提供了一个求解线性规划的函数 `linprog`。这个函数集中了求解线性规划的常用算法，如单纯形法和内点法，会根据问题的规模或用户的指定选择算法进行求解。  

SciPy 中线性规划模型的标准形为  
$$
\begin{align*}
\min &\quad z = c^{\mathrm{T}} x, \\
\text{s.t.} &\quad 
\begin{cases} 
A \cdot x \leqslant b, \\
\text{Aeq} \cdot x = \text{beq}, \\
\text{Lb} \leqslant x \leqslant \text{Ub}.
\end{cases}
\end{align*}
$$  


`linprog` 的基本调用格式为  
```python
from scipy.optimize import linprog

# 默认每个决策变量下界为0, 上界为+∞
res = linprog(c, A_ub=None, b_ub=None, A_eq=None, b_eq=None, bounds=(0, None), method='highs', callback=None, options=None, x0=None, integrality=None)

from scipy.optimize import linprog
c = [-1, 4]
A = [[-3, 1], [1, 2]]
b = [6, 4]
x0_bounds = (None, None)
x1_bounds = (-3, None)
res = linprog(c, A_ub=A, b_ub=b, bounds=[x0_bounds, x1_bounds])
print(res.fun)  # -22.0
print(res.x)   # [10., -3.]
print(res.message)  # 'Optimization terminated successfully. (HiGHS Status 7: Optimal)'

参数

c：1维数组  
线性目标函数的系数，表示要最小化的 $   c^T x   $。  
A_ub：2维数组，可选  
不等式约束矩阵。A_ub 的每一行指定一个线性不等式约束的系数。  
b_ub：1维数组，可选  
不等式约束向量。每个元素表示对应 $   A_{ub} @ x   $ 的上界。  
A_eq：2维数组，可选  
等式约束矩阵。A_eq 的每一行指定一个线性等式约束的系数。  
b_eq：1维数组，可选  
等式约束向量。A_eq @ x 的每个元素必须等于 b_eq 的对应元素。  
bounds：序列，可选  
为每个决策变量 $   x_i   $ 指定 (min, max) 的边界对，定义变量的最小和最大值。
如果提供单个 (min, max) 元组，则该边界应用于所有变量。  
使用 None 表示无界。例如，默认边界 (0, None) 表示所有变量非负，(None, None) 表示变量可以是任意实数。  

method：字符串，可选  
用于求解标准形式问题的算法。可选值包括：  
'highs'（默认）：自动选择 HiGHS 求解器。  
'highs-ds'：HiGHS 双阶段单纯形法。  
'highs-ipm'：HiGHS 内点法。  
'interior-point'（旧版，弃用）。  
'revised simplex'（旧版，弃用）。  
'simplex'（旧版，弃用）。  
旧版方法（'interior-point', 'revised simplex', 'simplex'）已在 SciPy 1.11.0 中弃用，将在未来移除。  

callback：可调用函数，可选  
如果提供回调函数，算法每迭代一次至少调用一次。回调函数需接受一个 scipy.optimize.OptimizeResult 对象，包含以下字段：  
x：当前解向量。  
fun：当前目标函数值 $   c @ x   $。  
success：布尔值，算法是否成功完成。  
slack：不等式约束的松弛变量值，$   b_{ub} - A_{ub} @ x   $。  
con：等式约束的残差，$   b_{eq} - A_{eq} @ x   $。  
phase：当前算法阶段。  
status：算法状态码：  
0：正常进行优化。  
1：达到迭代次数限制。  
2：问题可能不可行。  
3：问题可能无界。  
4：遇到数值困难。  
nit：当前迭代次数。  
message：算法状态描述。  
注意：HiGHS 方法当前不支持回调函数。

options：字典，可选  
求解器选项字典。所有方法支持以下选项：  
maxiter：最大迭代次数，默认值见方法文档。  
disp：布尔值，设为 True 打印收敛信息，默认 False。  
presolve：布尔值，设为 False 禁用自动预处理，默认 True。  
除 HiGHS 方法外，其他方法还支持：  
tol：浮点数，判断残差是否足够接近零的容差。  
autoscale：布尔值，设为 True 自动均衡化，适用于约束数值跨度大的情况，默认 False。  
rr：布尔值，设为 False 禁用自动冗余移除，默认 True。  
rr_method：字符串，指定冗余移除方法（仅限密集输入）：  
"SVD"：基于奇异值分解检测冗余行。  
"pivot"：基于文献 [5] 的算法。  
"ID"：随机插值分解。  
None：根据矩阵秩选择 "SVD" 或 "pivot"。  
默认：None。对于稀疏输入，使用基于 pivot 的算法。  

方法特定选项见 show_options('linprog')。  
x0：1维数组，可选  
决策变量的初始猜测值，仅用于 'revised simplex' 方法，且需为基本可行解。  
integrality：1维数组或整数，可选  
指定每个决策变量的整数约束类型：  
0：连续变量，无整数约束。  
1：整数变量，需在 bounds 内取整数。  
2：半连续变量，需在 bounds 内或取 0。  
3：半整数变量，需在 bounds 内取整数或取 0。  
默认：所有变量为连续变量。  
仅用于 'highs' 方法，其他方法忽略。  
返回值
res：OptimizeResult 对象，包含以下字段（返回值类型可能因优化是否成功而异，建议先检查 status）：

x：1维数组，最优解向量。
fun：浮点数，目标函数的最优值 $ c @ x $。
slack：1维数组，不等式约束的松弛变量值，$ b_{ub} - A_{ub} @ x $。
con：1维数组，等式约束的残差，$ b_{eq} - A_{eq} @ x $。
success：布尔值，算法是否成功找到最优解。
status：整数，算法退出状态：

0：优化成功。
1：达到迭代限制。
2：问题可能不可行。
3：问题可能无界。
4：遇到数值困难。


nit：总迭代次数。
message：字符串，算法退出状态描述。
```  

其中，`c` 对应于上述标准形中的目标向量，`A, b` 对应于不等号约束，`Aeq, beq` 对应于等号约束，`bounds` 是决策向量的下界向量和上界向量所组成的 $ n $ 个元素的元组，下面通过例子说明 `bounds` 的写法，`bounds` 的默认取值下界都是 $ 0 $，上界都是 $ +\infty $；返回值 `res.x` 是求得的最优解，`res.fun` 是目标函数的最优值.  


读者可以按如下方式看 `linprog` 函数的帮助.  
```python
>>> from scipy.optimize import linprog
>>> help(linprog)
```  


下面给出 `linprog` 函数帮助中的一个例子.  

**例 5.1** 求解线性规划问题  
$$
\begin{align*}
\min &\quad z = -x_1 + 4x_2, \\
\text{s.t.} &\quad 
\begin{cases} 
-3x_1 + x_2 \leqslant 6, \\
x_1 + 2x_2 \leqslant 4, \\
x_2 \geqslant -3.
\end{cases}
\end{align*}
$$  


上述模型中 $ x_1 $ 的取值是任意的，即 $ -\infty < x_1 < +\infty $，SciPy 中 $ x_1.\text{bounds} = (x_{1.\min}, x_{1.\max}) = (\text{None}, \text{None}) $；$ x_2 $ 的取值范围为 $ -3 \leqslant x_2 < +\infty $，SciPy 中 $ x_2.\text{bounds} = (x_{2.\min}, x_{2.\max}) = (-3, \text{None}) $. 因而在 Python 中上述问题的 $bounds = ((\text{None}, \text{None}), (-3, \text{None}))$.  

In [7]:
from scipy.optimize import linprog

c = [-1, 4]  # 价值向量
A = [[-3, 1], [1, 2]]  # 不等式约束的系数矩阵
b = [6, 4]  # 资源向量
x1_bounds = (None, None)
x2_bounds = (-3, None)
res = linprog(c, A_ub=A, b_ub=b, A_eq=None, b_eq=None, bounds=[x1_bounds, x2_bounds], method='highs')

print(f"目标函数的最小值: {res.fun}")
print(f"最优解为: {res.x}")

目标函数的最小值: -22.0
最优解为: [10. -3.]


In [1]:
from scipy.optimize import linprog

c = [-1, 4]  # 价值向量
A = [[-3, 1], [1, 2]]  # 不等式约束的系数矩阵
b = [6, 4]  # 资源向量
x1_bounds = (None, None)
x2_bounds = (-3, None)
res = linprog(c, A_ub=A, b_ub=b, A_eq=None, b_eq=None, bounds=[x1_bounds, x2_bounds], method='simplex')

print(f"目标函数的最小值: {res.fun}")
print(f"最优解为: {res.x}")

目标函数的最小值: -22.0
最优解为: [10. -3.]


C:\Users\yuanx\AppData\Local\Temp\ipykernel_17784\503506447.py:8: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(c, A_ub=A, b_ub=b, A_eq=None, b_eq=None, bounds=[x1_bounds, x2_bounds], method='simplex')


### 例 5.2 解析  
**题目**：求解线性规划问题，目标函数为最大化 $ z = x_1 - 2x_2 - 3x_3 $，约束条件含不等式（$\leq$、$\geq$）、等式，变量 $ x_1 $ 有下界、$ x_2 $ 非负、$ x_3 $ 无约束 。  

**转化标准形步骤**：  
1. **目标函数转换**：因为 SciPy 线性规划默认求最小化，最大化 $ z $ 等价于最小化 $ w = -z = -x_1 + 2x_2 + 3x_3 $ 。  
2. **约束条件调整**：  
    - 原不等式 $ -3x_1 + x_2 + 2x_3 \geq 4 $，两边乘 $-1$ 转化为 $ 3x_1 - x_2 - 2x_3 \leq -4 $（将 “$\geq$” 约束转为 “$\leq$” 形式，适配标准形 ）。  
    - 等式约束 $ 4x_1 - 2x_2 - x_3 = -6 $ 保持不变 。  
    - 变量范围：$ x_1 \geq -10 $、$ x_2 \geq 0 $、$ x_3 $ 无约束（标准形中通过 `bounds` 参数设置 ）。  

转化后标准形为：  
$$
\begin{align*}
\min &\quad w = -x_1 + 2x_2 + 3x_3 \\
\text{s.t.} &\quad 
\begin{cases} 
-2x_1 + x_2 + x_3 \leq 9 \\
3x_1 - x_2 - 2x_3 \leq -4 \\
4x_1 - 2x_2 - x_3 = -6 \\
x_1 \geq -10,\ x_2 \geq 0,\ x_3 \text{ 取值无约束}
\end{cases}
\end{align*}
$$  

后续可基于 SciPy 的 `linprog` 函数，按标准形配置参数（目标向量 `c`、不等式约束 `A, b`、等式约束 `Aeq, beq`、变量 bounds 等 ）求解，得到原问题的最优解（原问题最优解对应标准形最优解，目标函数值取反 ）。  

简言之，核心是把最大化、不等式方向、无约束变量等，转换为 SciPy 线性规划标准形（最小化、“$\leq$” 约束为主 ），再用工具函数求解 。

In [11]:
from scipy.optimize import linprog

c = [-1, 2, 3]
A = [[-2, 1, 1], [3, -1, -2]]
b = [[9], [-4]]  # 二维
Aeq = [[4, -2, -1]]  # 二维
beq = [-6]
x1_bounds = (-10, None)
x2_bounds = (0, None)
x3_bounds = (None, None)
res = linprog(c, A_ub=A, b_ub=b, A_eq=Aeq, b_eq=beq, bounds=[x1_bounds, x2_bounds, x3_bounds], method='highs')

print(f"目标函数的最小值: {res.fun}")
print(f"最优解为: {res.x}")

目标函数的最小值: 0.399999999999999
最优解为: [-1.6  0.  -0.4]


In [20]:
from scipy.optimize import linprog

c = [-1, 2, 3]
A_ub = [[-2, 1, 1], [3, -1, -2]]
b_ub = [[9], [-4]]  # 二维
A_eq = [[4, -2, -1]]  # 二维
b_eq = [-6]
LB = [-10, 0, None]
UB = [None] * len(c)  # 生成3个None的列表
bound = list(zip(LB, UB))  # 生成决策向量界限的列表
res = linprog(c, A_ub, b_ub, A_eq, b_eq, bound)

print(f"目标函数的最小值: {res.fun}")
print(f"最优解为: {res.x}")

目标函数的最小值: 0.399999999999999
最优解为: [-1.6  0.  -0.4]


In [17]:
from scipy.optimize import linprog

c = [-1, 2, 3]  # 目标函数系数
A_ub = [[-2, 1, 1], [3, -1, -2]]  # 不等式约束矩阵
b_ub = [9, -4]  # 不等式约束右端（一维数组）
A_eq = [[4, -2, -1]]  # 等式约束矩阵
b_eq = [-6]  # 等式约束右端
LB = [-10, 0, None]  # 下界
UB = [None, None, None]  # 上界
bound = tuple(zip(LB, UB))  # 边界：((-10, None), (0, None), (None, None))
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bound, method='highs')

print(f"目标函数的最小值: {res.fun}")
print(f"最优解为: {res.x}")
print(f"状态: {res.status}, 信息: {res.message}")

目标函数的最小值: 0.399999999999999
最优解为: [-1.6  0.  -0.4]
状态: 0, 信息: Optimization terminated successfully. (HiGHS Status 7: Optimal)


In [23]:
# 列表不能取相反数，可以利用数组
import numpy as np
from scipy.optimize import linprog

c = np.array([1, -2, -3])  # 数组方便下面取相反数
A = [[-2, 1, 1], [3, -1, -2]]
b = [[9], [-4]]
Aeq = [[4, -2, -1]]
beq = [-6]
LB = [-10, 0, None]
UB = [None] * len(c)
bound = list(zip(LB, UB))
res = linprog(-c, A, b, Aeq, beq, bound)

print(f"目标函数的最小值: {res.fun}")
print(f"最优解: {res.x}")

目标函数的最小值: 0.399999999999999
最优解: [-1.6  0.  -0.4]


例 5.3 加工一种食用油需要精炼若干种原料油并把它们混合起来。原料油的来源有两类共 5 种：植物油 VEG1、植物油 VEG2、非植物油 OIL1、非植物油 OIL2、非植物油 OIL3。购买每种原料油的价格（英镑/吨）如表 5.1 所示，最终产品以 150 英镑/吨的价格出售。植物油和非植物油需要在不同的生产线上进行精炼。每月能够精炼的植物油不超过 200 吨，非植物油不超过 250 吨；在精炼过程中，重量没有损失，精炼费用可忽略不计。最终产品要符合硬度的技术条件。按照硬度计量单位，它必须为 3～6。假定硬度的混合是线性的，而原材料的硬度如表 5.2 所示。  

为使利润最大，应该怎样指定它的月采购和加工计划。  


### 表 5.1 原料油价格  

| 原料油 | VEG1 | VEG2 | OIL1 | OIL2 | OIL3 |
| ---- | ---- | ---- | ---- | ---- | ---- |
| 价格 | 110 | 120 | 130 | 110 | 115 |  


### 表 5.2 原料油硬度表  

| 原料油 | VEG1 | VEG2 | OIL1 | OIL2 | OIL3 |
| ---- | ---- | ---- | ---- | ---- | ---- |
| 硬度值 | 8.8 | 6.1 | 2.0 | 4.2 | 5.0 |  


**解** 设 $ x_1, x_2, \cdots, x_5 $ 为每月需要采购的 5 种原料油吨数，$ x_6 $ 为每月加工的成品油吨数.  

#### (1) 目标函数是使净利润  
$$
z = -110x_1 - 120x_2 - 130x_3 - 110x_4 - 115x_5 + 150x_6
$$  
达到最大值.  


#### (2) 约束条件分为如下 4 类.  
##### (i) 精炼能力限制：  
- 植物油的精炼能力限制：$ x_1 + x_2 \leqslant 200 $;  
- 非植物油的精炼能力限制：$ x_3 + x_4 + x_5 \leqslant 250 $.  

##### (ii) 硬度限制：  
- 硬度上限的限制：$ 8.8x_1 + 6.1x_2 + 2.0x_3 + 4.2x_4 + 5.0x_5 \leqslant 6x_6 $;  
- 硬度下限的限制：$ 8.8x_1 + 6.1x_2 + 2.0x_3 + 4.2x_4 + 5.0x_5 \geqslant 3x_6 $.  

##### (iii) 均衡性限制  
$$
x_1 + x_2 + x_3 + x_4 + x_5 = x_6.
$$  

##### (iv) 非负性限制  
$$
x_i \geqslant 0,\quad i = 1,2,\cdots,6.
$$  


综上所述，建立如下的线性规划模型  
$$
\begin{align*}
\max &\quad z = -110x_1 - 120x_2 - 130x_3 - 110x_4 - 115x_5 + 150x_6, \\
\text{s.t.} &\quad 
\begin{cases} 
x_1 + x_2 \leqslant 200, \\
x_3 + x_4 + x_5 \leqslant 250, \\
8.8x_1 + 6.1x_2 + 2.0x_3 + 4.2x_4 + 5.0x_5 \leqslant 6x_6, \\
8.8x_1 + 6.1x_2 + 2.0x_3 + 4.2x_4 + 5.0x_5 \geqslant 3x_6, \\
x_1 + x_2 + x_3 + x_4 + x_5 = x_6, \\
x_i \geqslant 0,\quad i = 1,2,\cdots,6.
\end{cases}
\end{align*}
$$  


利用 Python 软件求解上述线性规划模型时，需要将其改写为 Python 的标准形：  
$$
\begin{align*}
\min &\quad w = 110x_1 + 120x_2 + 130x_3 + 110x_4 + 115x_5 - 150x_6, \\
\text{s.t.} &\quad 
\begin{cases} 
x_1 + x_2 \leqslant 200, \\
x_3 + x_4 + x_5 \leqslant 250, \\
8.8x_1 + 6.1x_2 + 2.0x_3 + 4.2x_4 + 5.0x_5 - 6x_6 \leqslant 0, \\
-8.8x_1 - 6.1x_2 - 2.0x_3 - 4.2x_4 - 5.0x_5 + 3x_6 \leqslant 0, \\
x_1 + x_2 + x_3 + x_4 + x_5 - x_6 = 0, \\
x_i \geqslant 0,\quad i = 1,2,\cdots,6.
\end{cases}
\end{align*}
$$  


调用 `linprog` 函数可求得月采购与生产计划如表 5.3 所示.  


### 表 5.3 月采购与生产计划  

| 原料油 | VEG1 | VEG2 | OIL1 | OIL2 | OIL3 |
| ---- | ---- | ---- | ---- | ---- | ---- |
| 采购量/吨 | 159.2593 | 40.7407 | 0 | 250 | 0 |  
| 生产量/吨 | \multicolumn{2}{c}{450} | \multicolumn{3}{c}{利润/英镑 $ 1.7593 \times 10^4 $} |

In [28]:
from scipy.optimize import linprog

c = [110, 120, 130, 110, 115, -150]
A = [[1, 1, 0, 0, 0, 0], [0, 0, 1, 1, 1, 0], [8.8, 6.1, 2.0, 4.2, 5.0, -6], [-8.8, -6.1, -2.0, -4.2, -5.0, 3]]
b = [[200], [250], [0], [0]]
Aeq = [[1, 1, 1, 1, 1, -1]]
beq = [0]
LB = [0] * len(c)
UB = [None] * len(c)
bound = list(zip(LB, UB))
res = linprog(c, A, b, Aeq, beq, bound)

print(f"目标函数的最小值: {res.fun}")
print(f"最优解: {res.x}")

目标函数的最小值: -17592.592592592595
最优解: [159.25925926  40.74074074   0.         250.           0.
 450.        ]


### `cvxopt.solvers` 模块求解线性规划
`cvxopt.solvers` 模块求解线性规划模型的标准形如下：  
$$
\begin{align*}
\min &\quad z = \boldsymbol{c}^{\mathrm{T}} \boldsymbol{x}, \\
\text{s.t.} &\quad 
\begin{cases} 
\boldsymbol{A} \cdot \boldsymbol{x} \leqslant \boldsymbol{b}, \\
\text{Aeq} \cdot \boldsymbol{x} = \text{beq}.
\end{cases}
\end{align*}
$$  


**例 5.4** 求解线性规划  
$$
\begin{align*}
\min &\quad z = -4x_1 - 5x_2, \\
\text{s.t.} &\quad 
\begin{cases} 
2x_1 + x_2 \leqslant 3, \\
x_1 + 2x_2 \leqslant 3, \\
x_1 \geqslant 0,\ x_2 \geqslant 0.
\end{cases}
\end{align*}
$$  


**解** 利用 Python 软件（结合 `cvxopt.solvers` 等工具 ）求得上述线性规划的最优解为 $ x_1 = 1, x_2 = 1 $；目标函数的最优值为 $ -9 $（因目标函数是 minimization 形式，实际原问题若理解为最大化需转换，此处按标准形求解结果 ）。  

简言之，先按 `cvxopt` 线性规划标准形（最小化、不等式与等式约束 ）整理问题，再用其求解函数运算，得到变量取值和最优目标值 。

In [ ]:
from cvxopt import matrix, solvers  # 必须写成double类型的数据，不然会抛出错误

c = matrix([-4.0, -5.0])
# G 必须是 4×2 矩阵（4 个约束，2 个变量）
# 注意：按列存储 (先写第一列，再写第二列)
# matrix([...], (4,2)) 里的 list 是 列优先，即先写完第一列，再写第二列
A = matrix([
    2.0, 1.0, -1.0, 0.0,
    1.0, 2.0, 0.0, -1.0
], (4, 2))
b = matrix([3.0, 3.0, 0.0, 0.0])
sol = solvers.lp(c, A, b)  # lp即linear programming

print(f"最优解为:\n{sol['x']}")
print(f"最优值为: {sol['primal objective']}")

     pcost       dcost       gap    pres   dres   k/t
 0: -8.1000e+00 -1.8300e+01  4e+00  0e+00  8e-01  1e+00
 1: -8.8055e+00 -9.4357e+00  2e-01  2e-16  4e-02  3e-02
 2: -8.9981e+00 -9.0049e+00  2e-03  6e-16  5e-04  4e-04
 3: -9.0000e+00 -9.0000e+00  2e-05  1e-16  5e-06  4e-06
 4: -9.0000e+00 -9.0000e+00  2e-07  1e-16  5e-08  4e-08
Optimal solution found.
最优解为:
[ 1.00e+00]
[ 1.00e+00]

最优值为: -8.999999811406719


**例 5.5** 求解线性规划  
$$
\begin{align*}
\max &\quad z = -2x_1 - x_2, \\
\text{s.t.} &\quad 
\begin{cases} 
-x_1 + x_2 \leqslant 1, \\
x_1 + x_2 \geqslant 2, \\
x_1 - 2x_2 \leqslant 4, \\
x_2 \geqslant 0, \\
x_1 + 2x_2 = 3.5.
\end{cases}
\end{align*}
$$  


**解** 标准形为  
$$
\begin{align*}
\min &\quad z = 2x_1 + x_2, \\
\text{s.t.} &\quad 
\begin{cases} 
-x_1 + x_2 \leqslant 1, \\
-x_1 - x_2 \leqslant -2, \\
x_1 - 2x_2 \leqslant 4, \\
-x_2 \leqslant 0, \\
x_1 + 2x_2 = 3.5.
\end{cases}
\end{align*}
$$  


利用 Python 软件求得最优解为 $ x_1 = 0.5, x_2 = 1.5 $；目标函数的最优值为 $ 2.5 $.

In [ ]:
from cvxopt import matrix, solvers  # 所有数据必须写成double类型的数据，不然会抛出错误

c = matrix([2.0, 1.0])
A = matrix([
    -1.0, -1.0, 1.0, 0.0,
    1.0, -1.0, -2.0, -1.0
], (4, 2))
b = matrix([1.0, -2.0, 4.0, 0.0])
Aeq = matrix([
    1.0, 2.0
], (1, 2))
beq = matrix([3.5])
sol = solvers.lp(c, A, b, Aeq, beq)

print(f"最优解为:\n{sol['x']}")
print(f"最优值为: {sol['primal objective']}")

     pcost       dcost       gap    pres   dres   k/t
 0:  5.5556e+00  1.2222e+00  1e+01  0e+00  2e+00  1e+00
 1:  4.6038e+00  3.7995e+00  2e+00  1e-16  4e-01  2e-01
 2:  2.5229e+00  2.4639e+00  2e-01  2e-16  4e-02  4e-02
 3:  2.5002e+00  2.4997e+00  2e-03  1e-16  4e-04  4e-04
 4:  2.5000e+00  2.5000e+00  2e-05  3e-16  4e-06  4e-06
 5:  2.5000e+00  2.5000e+00  2e-07  2e-16  4e-08  4e-08
Optimal solution found.
最优解为:
[ 5.00e-01]
[ 1.50e+00]

最优值为: 2.5000000246110483


首先介绍求解凸优化的 cvxpy 库.  

cvxpy 与 MATLAB 中 cvx 的工具库类似，用于求解凸优化问题. cvx 与 cvxpy 都是由 CIT 的 Stephen Boyd 教授课题组开发的. cvx 是用于 MATLAB 的库，cvxpy 是用于 Python 的库. 下载、安装及学习地址如下：  

cvx: http://cvxr.com/cvx/; cvxpy: http://www.cvxpy.org/.  

已安装的求解器:   
['CLARABEL', 'CVXOPT', 'GLPK', 'GLPK_MI', 'OSQP', 'SCIPY', 'SCS']
CLARABEL: Clarabel（一款开源的内点法求解器，支持线性规划和二次锥规划）  
CVXOPT: Convex Optimization（Python的凸优化库，支持线性、二次和锥规划）  
GLPK: GNU Linear Programming Kit（GNU线性规划工具包，支持线性规划）  
GLPK_MI: GNU Linear Programming Kit Mixed Integer（GLPK的混合整数线性规划扩展）  
OSQP: Operator Splitting Quadratic Program（基于算子分裂的二次规划求解器）  
SCIPY: Scientific Python（SciPy库中的优化模块，支持基础线性规划）  
SCS: Splitting Conic Solver（基于分裂锥的求解器，支持锥优化问题）

线性规划问题（LP）：GLPK, CBC, MOSEK 等。

二次规划问题（QP）：ECOS, OSQP, SCS 等。

整数规划问题（MIP）：GLPK_MI, CBC。

大规模问题：SCS, OSQP 等。

一般凸优化问题：ECOS, SCS。

**例 5.6** 已知某种商品 6 个仓库的存货量，8 个客户对该商品的需求量，单位商品运价如表 5.4 所示. 试确定 6 个仓库到 8 个客户的商品调运数量，使总的运输费用最小.  


### 表 5.4 单位商品运价表  

| 单位运价\客户\仓库 | V1 | V2 | V3 | V4 | V5 | V6 | V7 | V8 | 存货量 |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| W1 | 6 | 2 | 6 | 7 | 4 | 2 | 5 | 9 | 60 |
| W2 | 4 | 9 | 5 | 3 | 8 | 5 | 8 | 2 | 55 |
| W3 | 5 | 2 | 1 | 9 | 7 | 4 | 3 | 3 | 51 |
| W4 | 7 | 6 | 7 | 3 | 9 | 2 | 7 | 1 | 43 |
| W5 | 2 | 3 | 9 | 5 | 7 | 2 | 6 | 5 | 41 |
| W6 | 5 | 5 | 2 | 2 | 8 | 1 | 4 | 3 | 52 |
| 需求量 | 35 | 37 | 22 | 32 | 41 | 32 | 43 | 38 |  |  


**解** 设 $ x_{ij} (i = 1,2,\cdots,6; j = 1,2,\cdots,8) $ 表示第 $ i $ 个仓库运到第 $ j $ 个客户的商品数量，$ c_{ij} $ 表示第 $ i $ 个仓库到第 $ j $ 个客户的单位运价，$ d_j $ 表示第 $ j $ 个客户的需求量，$ e_i $ 表示第 $ i $ 个仓库的存货量，建立如下线性规划模型：  
\[
\begin{align*}
\min &\quad \sum_{i=1}^{6} \sum_{j=1}^{8} c_{ij} x_{ij}, \\
\text{s.t.} &\quad 
\begin{cases} 
\sum_{j=1}^{8} x_{ij} \leqslant e_i, \quad i = 1,2,\cdots,6, \\
\sum_{i=1}^{6} x_{ij} = d_j, \quad j = 1,2,\cdots,8, \\
x_{ij} \geqslant 0, \quad i = 1,2,\cdots,6; j = 1,2,\cdots,8.
\end{cases}
\end{align*}
\]  


利用 cvxpy 库，求得目标函数的最优值为 664，问题的最优解就不给出了. 把表 5.4 中的 7 行 9 列数据保存到 Excel 文件 Pdata5_6.xlsx 中.

In [ ]:
import pandas as pd
import cvxpy as cp

d1 = pd.read_excel("Pdata5_6.xlsx", header=None)
d2 = d1.values
c = d2[:-1, :-1]  # 运费价格(6, 8)
d = d2[-1, :-1].reshape(1, -1)  # 二维行向量, 顾客需求量(8,)
e = d2[:-1, -1].reshape(-1, 1)  # 二维列向量, 仓库存储量(6,)

x = cp.Variable((6, 8))  # Variable((6, 8), var1),创建矩阵变量(6, 8)
obj = cp.Minimize(cp.sum(cp.multiply(c, x)))  # multiply乘, sum求和，即最小化c和x的乘积之和，也就是目标函数
con = [cp.sum(x, axis=1, keepdims=True) <= e, cp.sum(x, axis=0, keepdims=True)==d, x>=0]  # 构造约束条件，keepdims=True保持输出维度，便于比较,axis=1代表列,行求和,axis=0代表行,列求和
prob = cp.Problem(obj, con)  # 构造模型，即构造优化问题，把目标函数obj和约束条件con放在一起
prob.solve(solver=cp.GLPK_MI, verbose=True)  # 求解模型，solver指定求解器，verbose（可选）：布尔值，True时输出求解器详细信息（如编译和求解过程）。

print(f"最优值为: {prob.value}")
print(f"最优解为:\n{x.value}")

(CVXPY) Aug 25 02:06:54 PM: Your problem has 48 variables, 62 constraints, and 0 parameters.
(CVXPY) Aug 25 02:06:54 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 25 02:06:54 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 25 02:06:54 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 25 02:06:54 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Aug 25 02:06:54 PM: Compiling problem (target solver=GLPK_MI).
(CVXPY) Aug 25 02:06:54 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> GLPK_MI
(CVXPY) Aug 25 02:06:54 PM: Applying reduction Dcp2Cone
(CVXPY) Aug 25 02:06:54 PM: Applying reduction CvxAttr2Constr
(CVXPY) Aug 25 02:06:54 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Aug 25 02:06:54 PM: Applying reduction GLPK_MI
(CVXPY) Aug 25 02:06:54 PM: Finished problem compilation

                                     CVXPY                                     
                                     v1.7.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
最优值为: 664.0
最优解为:
[[-0. 19. -0. -0. 41. -0. -0. -0.]
 [-0. -0. -0. 32. -0. -0. -0.  1.]
 [-0. 12. 22. -0. -0. -0. 17. -0

### 注 5.2  
在上面程序的目标函数中，我们需要的是两个矩阵对应元素相乘，使用的函数是 `cvxpy.multiply`。使用 `help(cvxpy.multiply)` 可以看到该函数的帮助，该函数在模块  

```python
cvxpy.atoms.affine.binary_operators
```  

中，调用该函数时，没有必要指明具体的模块，使用 `cvxpy.multiply` 调用就可以了。Python 的一些其他第三方库的部分函数调用也可以用类似的方式，即库名.函数（或别名.函数）的方式，不需要指明内部模块的名称。当然先加载具体的模块，再调用该模块下的函数也是可以的。